## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import openpyxl

warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Load Data

In [ ]:
# Load CSV file
df_bdcom = pd.read_csv('data/bronze/public_service_data/BDCOM_2023.csv')

# Load Excel file
df_od = pd.read_excel('data/bronze/public_service_data/BDCOM_2023_OD.xlsx')

print("BDCOM_2023.csv shape:", df_bdcom.shape)
print("BDCOM_2023_OD.xlsx shape:", df_od.shape)

## 3. Initial Data Inspection

In [ ]:
# Display first few rows of BDCOM data

print("BDCOM_2023.csv - First 5 rows")

display(df_bdcom.head())

In [ ]:
# Display first few rows of OD data
print("BDCOM_2023_OD.xlsx - First 5 rows")
display(df_od.head())

In [ ]:
# Basic info about BDCOM data
print("BDCOM_2023.csv - Data Info")
df_bdcom.info()

In [ ]:
# Basic info about OD data
print("BDCOM_2023_OD.xlsx - Data Info")
df_od.info()

## 4. Data Merging

Merge the two datasets based on the activity code field (`codact` from BDCOM matches with activity code from OD)

In [ ]:
# Rename the first column of df_od to match for merging
# The column "Code activité (224 postes)" should match with "codact" from df_bdcom
df_od_clean = df_od.rename(columns={'Code activité (224 postes)': 'codact'})

print("Columns in df_od after renaming:")
print(df_od_clean.columns.tolist())

In [ ]:
# Check unique values in merge key
print(f"Unique codact values in BDCOM: {df_bdcom['codact'].nunique()}")
print(f"Unique codact values in OD: {df_od_clean['codact'].nunique()}")

# Sample values
print("\nSample codact from BDCOM:")
print(df_bdcom['codact'].unique()[:10])
print("\nSample codact from OD:")
print(df_od_clean['codact'].unique()[:10])

In [ ]:
# Perform the merge
df_merged = pd.merge(
    df_bdcom,
    df_od_clean,
    on='codact',
    how='left',  # Keep all BDCOM records
    indicator=True  # Add merge indicator
)

print(f"Merged dataset shape: {df_merged.shape}")
print(f"\nMerge statistics:")
print(df_merged['_merge'].value_counts())

In [ ]:
# Display merged data sample

print("Merged Dataset - First 5 rows")

display(df_merged.head())

## 5. Exploratory Data Analysis (EDA)

### 5.1 Dataset Overview

In [ ]:

print("DATASET OVERVIEW")

print(f"Total Rows: {df_merged.shape[0]:,}")
print(f"Total Columns: {df_merged.shape[1]}")
print(f"\nMemory Usage: {df_merged.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Column names and data types
print("\nColumns and Data Types:")
print("-" * 80)
for col, dtype in df_merged.dtypes.items():
    print(f"{col:<40} {str(dtype):<20}")

### 5.2 Missing Values Analysis

In [ ]:
# Missing values count and percentage
missing_data = pd.DataFrame({
    'Column': df_merged.columns,
    'Missing_Count': df_merged.isnull().sum(),
    'Missing_Percentage': (df_merged.isnull().sum() / len(df_merged) * 100).round(2)
}).sort_values('Missing_Count', ascending=False)


print("MISSING VALUES ANALYSIS")

display(missing_data[missing_data['Missing_Count'] > 0])

In [ ]:
# Visualize missing values
missing_cols = missing_data[missing_data['Missing_Count'] > 0]['Column'].tolist()

if len(missing_cols) > 0:
    plt.figure(figsize=(12, 6))
    missing_plot_data = missing_data[missing_data['Missing_Count'] > 0].head(20)
    plt.barh(missing_plot_data['Column'], missing_plot_data['Missing_Percentage'])
    plt.xlabel('Missing Percentage (%)')
    plt.title('Top 20 Columns with Missing Values')
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found!")

### 5.3 Statistical Summary

In [ ]:
# Numeric columns summary

print("NUMERICAL FEATURES SUMMARY")

display(df_merged.describe())

In [ ]:
# Categorical columns summary
categorical_cols = df_merged.select_dtypes(include=['object']).columns


print("CATEGORICAL FEATURES SUMMARY")

for col in categorical_cols[:10]:  # Show first 10 categorical columns
    print(f"\n{col}:")
    print(f"\tUnique values: {df_merged[col].nunique()}")
    print(f"\tTop 5 values:")
    print(df_merged[col].value_counts().head())

### 5.4 Duplicate Analysis

In [ ]:
# Check for duplicates
duplicates = df_merged.duplicated().sum()

print("DUPLICATE ANALYSIS")

print(f"Total duplicate rows: {duplicates:,}")
print(f"Percentage: {(duplicates/len(df_merged)*100):.2f}%")

# Check for duplicates based on OBJECTID (should be unique)
if 'OBJECTID' in df_merged.columns:
    objectid_dupes = df_merged['OBJECTID'].duplicated().sum()
    print(f"\nDuplicate OBJECTID values: {objectid_dupes}")

### 5.5 Key Variables Distribution

In [ ]:
# Distribution of activity types
if 'TYPE' in df_merged.columns:
    
    print("DISTRIBUTION OF LOCAL TYPES")
    
    print(df_merged['TYPE'].value_counts())
    
    plt.figure(figsize=(10, 6))
    df_merged['TYPE'].value_counts().plot(kind='bar')
    plt.title('Distribution of Local Types')
    plt.xlabel('Type')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Distribution by activity levels (niv2, niv8, niv18, niv47)
activity_levels = ['niv2', 'niv8', 'niv18', 'niv47']
existing_levels = [col for col in activity_levels if col in df_merged.columns]

if existing_levels:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    axes = axes.ravel()
    
    for idx, col in enumerate(existing_levels):
        top_values = df_merged[col].value_counts().head(10)
        axes[idx].barh(range(len(top_values)), top_values.values)
        axes[idx].set_yticks(range(len(top_values)))
        axes[idx].set_yticklabels(top_values.index)
        axes[idx].set_xlabel('Count')
        axes[idx].set_title(f'Top 10 {col.upper()} Categories')
        axes[idx].invert_yaxis()
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Geographic distribution (X, Y coordinates)
if 'X' in df_merged.columns and 'Y' in df_merged.columns:
    plt.figure(figsize=(12, 8))
    plt.scatter(df_merged['X'], df_merged['Y'], alpha=0.3, s=1)
    plt.xlabel('X Coordinate')
    plt.ylabel('Y Coordinate')
    plt.title('Geographic Distribution of Points')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Surface area distribution
if 'surf' in df_merged.columns:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Histogram
    axes[0].hist(df_merged['surf'].dropna(), bins=50, edgecolor='black')
    axes[0].set_xlabel('Surface Area')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Surface Area')
    
    # Box plot
    axes[1].boxplot(df_merged['surf'].dropna())
    axes[1].set_ylabel('Surface Area')
    axes[1].set_title('Surface Area Box Plot')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nSurface Area Statistics:")
    print(df_merged['surf'].describe())

In [ ]:
# Top establishments (ens)
if 'ens' in df_merged.columns:
    
    print("TOP 20 ESTABLISHMENTS")
    
    top_ens = df_merged['ens'].value_counts().head(20)
    display(pd.DataFrame({'Establishment': top_ens.index, 'Count': top_ens.values}))
    
    plt.figure(figsize=(12, 6))
    top_ens.plot(kind='barh')
    plt.xlabel('Count')
    plt.title('Top 20 Establishments')
    plt.tight_layout()
    plt.show()

### 5.6 Correlation Analysis

In [ ]:
# Correlation matrix for numeric columns
numeric_cols = df_merged.select_dtypes(include=[np.number]).columns.tolist()

if len(numeric_cols) > 1:
    correlation_matrix = df_merged[numeric_cols].corr()
    
    plt.figure(figsize=(14, 10))
    sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
                center=0, square=True, linewidths=1)
    plt.title('Correlation Matrix of Numeric Features')
    plt.tight_layout()
    plt.show()

## 6. Data Cleaning

In [ ]:
# Create a copy for cleaning
df_clean = df_merged.copy()

print("Starting data cleaning process...")
print(f"Initial shape: {df_clean.shape}")

### 6.1 Remove Merge Indicator Column

In [ ]:
# Drop the merge indicator column
if '_merge' in df_clean.columns:
    df_clean = df_clean.drop('_merge', axis=1)
    print("Dropped '_merge' indicator column")

### 6.2 Handle Duplicates

In [ ]:
# Remove duplicate rows
initial_rows = len(df_clean)
df_clean = df_clean.drop_duplicates()
rows_removed = initial_rows - len(df_clean)

print(f"Removed {rows_removed:,} duplicate rows")
print(f"Remaining rows: {len(df_clean):,}")

### 6.3 Handle Missing Values

In [ ]:
# Strategy for handling missing values:
# 1. For categorical columns with few missing values: fill with 'Unknown' or mode
# 2. For numeric columns: consider median or mean based on distribution
# 3. For columns with high missing percentage (>50%): consider dropping

missing_threshold = 50  # Drop columns with >50% missing

# Identify columns to drop
cols_to_drop = missing_data[missing_data['Missing_Percentage'] > missing_threshold]['Column'].tolist()

if cols_to_drop:
    print(f"\nDropping {len(cols_to_drop)} columns with >{missing_threshold}% missing values:")
    for col in cols_to_drop:
        print(f"  - {col}: {missing_data[missing_data['Column']==col]['Missing_Percentage'].values[0]:.2f}% missing")
    df_clean = df_clean.drop(cols_to_drop, axis=1)

In [ ]:
# Fill missing values in remaining columns

# For 'let' (letter) column - fill with empty string
if 'let' in df_clean.columns:
    df_clean['let'].fillna('', inplace=True)
    print("Filled missing 'let' values with empty string")

# For 'cc_id' and 'cc_niv' - fill with 0 or appropriate default
for col in ['cc_id', 'cc_niv']:
    if col in df_clean.columns:
        df_clean[col].fillna(0, inplace=True)
        print(f"Filled missing '{col}' values with 0")

# For categorical activity columns - fill with 'Unknown'
categorical_activity_cols = ['Libellé activité (224 postes)', 'TYPE', 
                             'Libellé TYPE (local)', 'Libellé activité 47 postes',
                             'Libellé activité 18 postes', 'Libellé activité 8 postes)',
                             'Libellé activité 2 postes']

for col in categorical_activity_cols:
    if col in df_clean.columns and df_clean[col].isnull().any():
        df_clean[col].fillna('Unknown', inplace=True)
        print(f"Filled missing '{col}' values with 'Unknown'")

### 6.4 Data Type Corrections

In [ ]:
# Ensure proper data types

# Integer columns
int_cols = ['OBJECTID', 'c_ord', 'arro', 'qua', 'num', 'seq', 'bio', 'surf', 
            'cc_id', 'cc_niv', 'niv47', 'niv18', 'niv8', 'niv2',
            'Code activité 47 postes', 'Code activité 18 postes', 
            'Code activité 8 postes', 'Code activité 2 postes']

for col in int_cols:
    if col in df_clean.columns:
        try:
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').fillna(0).astype(int)
        except:
            print(f"Could not convert {col} to integer")

# Float columns (coordinates)
float_cols = ['X', 'Y', 'xbis', 'ybis']
for col in float_cols:
    if col in df_clean.columns:
        try:
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
        except:
            print(f"Could not convert {col} to float")

print("\nData type corrections completed")

### 6.5 String Data Cleaning

In [ ]:
# Clean string columns: strip whitespace, standardize case where appropriate
string_cols = df_clean.select_dtypes(include=['object']).columns

for col in string_cols:
    # Strip whitespace
    df_clean[col] = df_clean[col].astype(str).str.strip()
    
    # Remove extra spaces
    df_clean[col] = df_clean[col].str.replace(r'\s+', ' ', regex=True)

print(f"Cleaned {len(string_cols)} string columns")

### 6.6 Outlier Detection and Handling

In [ ]:
# Detect outliers using IQR method for numeric columns
def detect_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return len(outliers), lower_bound, upper_bound

# Check for outliers in surface area
if 'surf' in df_clean.columns:
    outlier_count, lower, upper = detect_outliers_iqr(df_clean, 'surf')
    print(f"\nOutliers in 'surf':")
    print(f"  Count: {outlier_count}")
    print(f"  Lower bound: {lower:.2f}")
    print(f"  Upper bound: {upper:.2f}")
    print(f"  Min value: {df_clean['surf'].min()}")
    print(f"  Max value: {df_clean['surf'].max()}")
    
    # Option: Cap outliers (uncomment if needed)
    # df_clean.loc[df_clean['surf'] < lower, 'surf'] = lower
    # df_clean.loc[df_clean['surf'] > upper, 'surf'] = upper

### 6.7 Final Data Quality Check

In [ ]:

print("FINAL DATA QUALITY REPORT")


print(f"\nFinal dataset shape: {df_clean.shape}")
print(f"Total rows: {len(df_clean):,}")
print(f"Total columns: {df_clean.shape[1]}")

# Missing values check
final_missing = df_clean.isnull().sum().sum()
print(f"\nTotal missing values: {final_missing:,}")

if final_missing > 0:
    print("\nColumns with remaining missing values:")
    remaining_missing = df_clean.isnull().sum()
    remaining_missing = remaining_missing[remaining_missing > 0].sort_values(ascending=False)
    for col, count in remaining_missing.items():
        pct = (count / len(df_clean)) * 100
        print(f"  {col}: {count:,} ({pct:.2f}%)")

# Duplicates check
final_dupes = df_clean.duplicated().sum()
print(f"\nDuplicate rows: {final_dupes}")

# Data types
print("\nData types distribution:")
print(df_clean.dtypes.value_counts())

In [ ]:
# Display sample of cleaned data
print("CLEANED DATA SAMPLE")

display(df_clean.head(10))

## 7. Save Cleaned Data

In [ ]:
# Save to CSV
output_csv = '/data/BDCOM_2023_merged_cleaned.csv'
df_clean.to_csv(output_csv, index=False, encoding='utf-8-sig')
print(f"Cleaned data saved to: {output_csv}")

# Save to Excel (optional)
output_xlsx = '/data/BDCOM_2023_merged_cleaned.xlsx'
df_clean.to_excel(output_xlsx, index=False, engine='openpyxl')
print(f"Cleaned data saved to: {output_xlsx}")

## 8. Summary Statistics of Cleaned Data

In [ ]:
# Generate comprehensive summary

print("CLEANED DATA SUMMARY")


summary_stats = {
    'Metric': [
        'Total Records',
        'Total Features',
        'Numeric Features',
        'Categorical Features',
        'Missing Values',
        'Duplicate Rows',
        'Memory Usage (MB)'
    ],
    'Value': [
        f"{len(df_clean):,}",
        df_clean.shape[1],
        len(df_clean.select_dtypes(include=[np.number]).columns),
        len(df_clean.select_dtypes(include=['object']).columns),
        f"{df_clean.isnull().sum().sum():,}",
        df_clean.duplicated().sum(),
        f"{df_clean.memory_usage(deep=True).sum() / 1024**2:.2f}"
    ]
}

summary_df = pd.DataFrame(summary_stats)
display(summary_df)

In [ ]:
# Activity distribution summary
if 'Libellé activité 8 postes)' in df_clean.columns:
    print("\n" + "=" * 80)
    print("ACTIVITY DISTRIBUTION (8 CATEGORIES)")
    
    activity_dist = df_clean['Libellé activité 8 postes)'].value_counts()
    activity_df = pd.DataFrame({
        'Activity': activity_dist.index,
        'Count': activity_dist.values,
        'Percentage': (activity_dist.values / activity_dist.sum() * 100).round(2)
    })
    display(activity_df)

## 9. Export Data Dictionary

In [ ]:
# Create data dictionary
data_dict = pd.DataFrame({
    'Column_Name': df_clean.columns,
    'Data_Type': df_clean.dtypes.values,
    'Non_Null_Count': df_clean.count().values,
    'Null_Count': df_clean.isnull().sum().values,
    'Unique_Values': [df_clean[col].nunique() for col in df_clean.columns],
    'Sample_Value': [str(df_clean[col].iloc[0]) if len(df_clean) > 0 else '' for col in df_clean.columns]
})

# Save data dictionary
data_dict_path = '/data/BDCOM_2023_data_dictionary.csv'
data_dict.to_csv(data_dict_path, index=False)
print(f"Data dictionary saved to: {data_dict_path}")

display(data_dict)